[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/24_ensemble_learning/01_voting_and_diversity.ipynb)

# Bài 01 — Voting và sự đa dạng

**🎯 Mục tiêu**
- Vì sao kết hợp nhiều model lại tốt hơn: toán của **bỏ phiếu đa số** (Condorcet) và vai trò của **tương quan** giữa các model.
- **Phân rã ambiguity** cho hồi quy: sai số của trung bình = sai số trung bình − độ đa dạng.
- **Hard voting vs soft voting**; đo độ đa dạng (bất đồng, tương quan lỗi); trọng số.
- Khi nào voting **làm hại**: thêm model yếu, model quá giống nhau.

> Bagging/Random Forest (chương 11) và boosting (chương 12) là ensemble **cùng loại** model. Chương này kết hợp các model **khác loại**.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/24_ensemble_learning

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import binom
from sklearn.compose import ColumnTransformer
from sklearn.datasets import load_digits
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestClassifier, RandomForestRegressor, VotingClassifier
from sklearn.linear_model import LogisticRegression, RidgeCV
from sklearn.metrics import accuracy_score, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC, SVR
from sklearn.tree import DecisionTreeClassifier

from ml_course.visualization import set_style

set_style()
rng = np.random.default_rng(0)

## 1. Bỏ phiếu đa số: toán của "trí tuệ đám đông"

$n$ model **độc lập**, mỗi model đúng với xác suất $p > 0.5$. Bỏ phiếu đa số đúng khi hơn nửa số model đúng:
$$P(\text{đa số đúng}) = \sum_{k > n/2}\binom nk p^k(1 - p)^{n-k}$$

In [ ]:
rows = []
for p in [0.55, 0.6, 0.7]:
    rows.append({"p (từng model)": p, **{f"n = {n}": binom.sf(n // 2, n, p) for n in [1, 5, 11, 25, 101]}})
pd.DataFrame(rows).set_index("p (từng model)").round(3)

Kết quả "kỳ diệu" (định lý bồi thẩm đoàn Condorcet) — nhưng dựa trên giả định **độc lập**. Các model train trên cùng dữ liệu thường **sai cùng chỗ**. Mô phỏng: lỗi của mỗi model gồm một phần **chung** (tương quan $\rho$) và một phần riêng:

In [ ]:
def majority_accuracy(n_models, p, rho, n_samples=20000, seed=0):
    r = np.random.default_rng(seed)
    shared = r.standard_normal(n_samples)[:, None]
    own = r.standard_normal((n_samples, n_models))
    latent = np.sqrt(rho) * shared + np.sqrt(1 - rho) * own           # tương quan ρ giữa các model
    correct = latent < np.quantile(latent, p)                          # mỗi model đúng với xác suất ≈ p
    return (correct.sum(1) > n_models / 2).mean()


rhos = [0, 0.2, 0.5, 0.8, 1.0]
for n in [5, 25]:
    print(f"{n:2d} model, p = 0.7 — accuracy đa số theo ρ:", {rho: round(majority_accuracy(n, 0.7, rho), 3) for rho in rhos})

Tương quan càng cao, lợi ích càng nhỏ; $\rho = 1$ (các model giống hệt) → không lợi gì. **Đa dạng** (diversity) là nguyên liệu chính của ensemble — đó là lý do Random Forest chọn feature ngẫu nhiên để giảm tương quan giữa các cây (chương 11).

## 2. Phân rã ambiguity (hồi quy)

Với ensemble trung bình $\bar f = \frac1M\sum_m f_m$, tại mọi điểm (Krogh & Vedelsby, 1995):
$$\underbrace{(\bar f - y)^2}_{\text{lỗi ensemble}} = \underbrace{\frac1M\sum_m(f_m - y)^2}_{\text{lỗi trung bình của các model}} - \underbrace{\frac1M\sum_m(f_m - \bar f)^2}_{\text{độ đa dạng (ambiguity)}}$$

Ensemble trung bình **không bao giờ tệ hơn** trung bình các thành viên, và tốt hơn đúng bằng mức đa dạng. Kiểm tra trên dữ liệu giá nhà:

In [ ]:
house = pd.read_csv("../../data/sample/house_prices.csv")
y_h = house.pop("price_billion_vnd").to_numpy()
num_cols = ["area_m2", "bedrooms", "bathrooms", "age_years", "distance_to_center_km"]


def prep():
    return ColumnTransformer([("num", StandardScaler(), num_cols), ("cat", OneHotEncoder(handle_unknown="ignore"), ["district"]),
                              ("bin", "passthrough", ["has_garden"])], sparse_threshold=0)


Xh_tr, Xh_te, yh_tr, yh_te = train_test_split(house, y_h, test_size=0.3, random_state=0)
regs = {"Ridge": make_pipeline(prep(), RidgeCV()), "KNN": make_pipeline(prep(), KNeighborsRegressor(10)),
        "SVR": make_pipeline(prep(), SVR(C=10)), "Random Forest": make_pipeline(prep(), RandomForestRegressor(300, random_state=0, n_jobs=-1)),
        "HistGB": make_pipeline(prep(), HistGradientBoostingRegressor(random_state=0))}
P = np.column_stack([m.fit(Xh_tr, yh_tr).predict(Xh_te) for m in regs.values()])
f_bar = P.mean(1)
ens_err = np.mean((f_bar - yh_te) ** 2)
avg_err = np.mean((P - yh_te[:, None]) ** 2)
ambiguity = np.mean((P - f_bar[:, None]) ** 2)
for name, j in zip(regs, range(P.shape[1])):
    print(f"{name:14s} MSE test = {np.mean((P[:, j] - yh_te) ** 2):.4f}")
print(f"\nlỗi trung bình các model = {avg_err:.4f} | độ đa dạng = {ambiguity:.4f} | hiệu = {avg_err - ambiguity:.4f}")
print(f"MSE của ensemble trung bình = {ens_err:.4f}  ← khớp đúng phân rã")

Ensemble trung bình tốt hơn **mọi** thành viên ở đây — vì các model mắc lỗi **khác nhau** (độ đa dạng lớn). Nếu một model vượt trội hẳn các model còn lại, trung bình đều có thể thua model tốt nhất — cần **trọng số** (bài 02–03).

## 3. Hard voting vs soft voting

- **Hard voting**: mỗi model bỏ một phiếu cho lớp dự đoán; chọn lớp nhiều phiếu nhất.
- **Soft voting**: trung bình **xác suất** dự đoán, chọn lớp có xác suất trung bình lớn nhất — tận dụng độ tự tin, thường tốt hơn **nếu** xác suất được hiệu chỉnh tốt (chương 37).

Dữ liệu chữ số viết tay (10 lớp):

In [ ]:
Xd, yd = load_digits(return_X_y=True)
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(Xd, yd, test_size=0.4, random_state=0, stratify=yd)
clfs = {
    "LogReg": make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)),
    "KNN": make_pipeline(StandardScaler(), KNeighborsClassifier(5)),
    "SVC": make_pipeline(StandardScaler(), SVC(probability=True, random_state=0)),
    "Random Forest": RandomForestClassifier(200, random_state=0),
    "Naive Bayes": GaussianNB(),
    "Decision Tree": DecisionTreeClassifier(random_state=0),
}
pred = {name: m.fit(Xd_tr, yd_tr).predict(Xd_te) for name, m in clfs.items()}
for name, p in pred.items():
    print(f"{name:14s} accuracy = {accuracy_score(yd_te, p):.4f}")

In [ ]:
combos = {"3 model mạnh (LogReg, KNN, SVC)": ["LogReg", "KNN", "SVC"],
          "+ Random Forest": ["LogReg", "KNN", "SVC", "Random Forest"],
          "+ 2 model yếu (NB, Tree)": ["LogReg", "KNN", "SVC", "Random Forest", "Naive Bayes", "Decision Tree"],
          "SVC + 2 model yếu": ["SVC", "Naive Bayes", "Decision Tree"]}
rows = []
for label, names in combos.items():
    row = {"tổ hợp": label, "tốt nhất trong nhóm": max(accuracy_score(yd_te, pred[n]) for n in names)}
    for voting in ["hard", "soft"]:
        vc = VotingClassifier([(n, clfs[n]) for n in names], voting=voting).fit(Xd_tr, yd_tr)
        row[voting] = accuracy_score(yd_te, vc.predict(Xd_te))
    rows.append(row)
pd.DataFrame(rows).set_index("tổ hợp").round(4)

- Ba model **mạnh và khác loại** → voting vượt model tốt nhất (soft > hard).
- Thêm model **yếu** (Naive Bayes, cây đơn ~0.82–0.84) kéo ensemble xuống — với soft voting, xác suất quá tự tin của Naive Bayes (chương 09) càng gây hại.
- Một model mạnh + hai model yếu: đa số phiếu thường là của hai model yếu → **tệ hơn hẳn** model mạnh đứng một mình.

## 4. Đo độ đa dạng

Hai model **sai ở những mẫu khác nhau** thì bổ sung cho nhau. Đo bằng tỉ lệ bất đồng (**disagreement**) và tương quan của vector lỗi:

In [ ]:
names = list(clfs)
errors = np.column_stack([(pred[n] != yd_te).astype(float) for n in names])
disagree = np.array([[np.mean(pred[a] != pred[b]) for b in names] for a in names])
err_corr = np.corrcoef(errors.T)
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
for ax, M, title in [(axes[0], disagree, "tỉ lệ bất đồng giữa các cặp model"), (axes[1], err_corr, "tương quan của lỗi")]:
    im = ax.imshow(M, cmap="viridis")
    ax.set_xticks(range(len(names)), names, rotation=45, ha="right")
    ax.set_yticks(range(len(names)), names)
    for i in range(len(names)):
        for j in range(len(names)):
            ax.text(j, i, f"{M[i, j]:.2f}", ha="center", va="center", color="w", fontsize=8)
    ax.set_title(title)
plt.tight_layout()
plt.show()

Bất đồng cao **không tự động tốt**: Naive Bayes và cây đơn bất đồng nhiều với mọi model vì chúng **sai nhiều**. Điều ta muốn: các model **đều giỏi** nhưng **sai ở chỗ khác nhau** (tương quan lỗi thấp giữa các model mạnh) — đúng với bộ ba LogReg/KNN/SVC.

## 5. Voting có trọng số

`VotingClassifier(weights=...)`: model tốt hơn có tiếng nói lớn hơn. Chọn trọng số trên **validation** (không phải test), hoặc để một meta-model học trọng số — đó là **stacking** (bài 02).

In [ ]:
X_fit, X_val, y_fit, y_val = train_test_split(Xd_tr, yd_tr, test_size=0.3, random_state=1, stratify=yd_tr)
members = ["LogReg", "KNN", "SVC", "Random Forest", "Naive Bayes", "Decision Tree"]
val_acc = np.array([accuracy_score(y_val, clfs[n].fit(X_fit, y_fit).predict(X_val)) for n in members])
weights = np.clip(val_acc - 0.85, 0, None)                              # model dưới ngưỡng 0.85 trên validation → trọng số 0
print("accuracy validation:", dict(zip(members, val_acc.round(3))))
print("trọng số:", dict(zip(members, (weights / weights.sum()).round(3))))
wv = VotingClassifier([(n, clfs[n]) for n in members], voting="soft", weights=weights).fit(Xd_tr, yd_tr)
print(f"soft voting có trọng số (6 model) — accuracy test = {accuracy_score(yd_te, wv.predict(Xd_te)):.4f}")

Trọng số "tắt" các model yếu và khôi phục lợi ích của ensemble mà không phải tự tay loại model.

## 🧠 Kiểm tra nhanh

1. Vì sao bỏ phiếu đa số của 25 model độc lập (p = 0.6) tốt hơn hẳn một model? Giả định nào hiếm khi đúng?
2. Viết phân rã ambiguity. Ensemble trung bình có thể tệ hơn model tệ nhất không? Tệ hơn model tốt nhất?
3. Khi nào soft voting tốt hơn hard voting? Khi nào nguy hiểm?
4. Vì sao thêm model yếu có thể làm ensemble tệ đi?
5. "Hai model bất đồng nhiều → kết hợp tốt" — đúng hay sai?

## 📌 Tóm tắt

- Ensemble mạnh khi thành viên **giỏi** và **đa dạng** (lỗi ít tương quan); tương quan cao → lợi ích biến mất.
- Hồi quy: lỗi ensemble = lỗi trung bình − độ đa dạng.
- Soft voting tận dụng xác suất (cần hiệu chỉnh); model yếu kéo ensemble xuống → loại bỏ hoặc trọng số.

## 📚 Tài liệu tham khảo

- Dietterich (2000) — *Ensemble Methods in Machine Learning*
- Krogh & Vedelsby (1995) — *Neural Network Ensembles, Cross Validation, and Active Learning*
- Kuncheva & Whitaker (2003) — *Measures of Diversity in Classifier Ensembles*
- Zhou (2012) — *Ensemble Methods: Foundations and Algorithms*